In [17]:
%load_ext autoreload
%autoreload 2

import os,sys
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
import util as yu
from util import *
import util_charge as yuc

yu.setpath('check_lbdfit_vs_2st')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
symmetrizeQ=True
extraLabel='' if symmetrizeQ else '_0sym'

ens='b'
path=f'/p/project1/ngff/li47/code/projectData/07_Nsgm/charges/{yu.ens2full[ens]}.h5'
[c2pt_disc,tfs_conn,tfs_disc,tf2c2pt_conn,key2tf2c3pt,key2tf2ratio]=yuc.load(path,symmetrizeQ=symmetrizeQ)

[ens2pars_jk_meff1st,ens2pars_jk_meff2st,ens2pars_jk_meff3st]=yu.load_pkl_reg('ens2pars_jk_meffnst_selected',pathlabel='analysis_c2pt')
pars_jk_meff2st=ens2pars_jk_meff2st[ens]

In [43]:
xunit=yu.ens2a[ens]
yunit=yu.ens2amul[ens]*yu.ens2aInv[ens]
tf2ratio=key2tf2ratio['gS+']

tfmins_2st=[8,10,12,14,16]
tcmins_2st=[1,2,3,4,5,6]
label2fits={}; label2fits_ds2={}; label2fits_unc={}
for label in ['2st2step_SYM','2st2step_SYMshare','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']:
    print(label)
    label2fits[label]=yu.doFits_3pt(label,tf2ratio,tfmins_2st,tcmins_2st,pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,label=label+extraLabel,verbose=1)
    label2fits_unc[label]=yu.doFits_3pt(label,tf2ratio,tfmins_2st,tcmins_2st[:2],pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,label=label+'_unc'+extraLabel,corrQ=False,verbose=1)
    label2fits_ds2[label]=yu.doFits_3pt(label,tf2ratio,tfmins_2st,tcmins_2st,pars_jk_meff2st=pars_jk_meff2st,symmetrizeQ=symmetrizeQ,label=label+'_ds2'+extraLabel,verbose=1,downSampling=[1,2])

2st2step_SYM
2st2step_SYMshare
2st2step_SYM_0ra11
2st2step_SYM_0rc1_0ra11
2st2step_SYM_share11


In [40]:
print(yu.ens2amul[ens],yu.ens2aInv[ens],yunit)

0.00072 2479.5777303003642 1.7852959658162624


In [46]:
print(f'[mN,dE1,rc1]={yu.jackme_un2str(pars_jk_meff2st)}')
for fit in label2fits['2st2step_SYM']:
    fitlabel,pars,*_=fit
    if fitlabel not in [(8,1),(8,2)]:
        continue
    t=pars.copy()
    t[:,2:]/=t[:,0:1]
    t[:,0]*=1.6555
    print(f'(ts_low,tc_cut)={fitlabel}',f'[sgm,dE1_3pt,ra01,ra11]={yu.jackme_un2str(t)}')

[mN,dE1,rc1]=[0.3809(13), 0.232(16), 0.846(21)]
(ts_low,tc_cut)=(8, 1) [sgm,dE1_3pt,ra01,ra11]=[39.9(2.3), 0.201(17), -0.567(19), -0.14(16)]
(ts_low,tc_cut)=(8, 2) [sgm,dE1_3pt,ra01,ra11]=[48.0(4.1), 0.148(16), -0.604(26), 0.16(13)]


In [21]:
n=2
fitlabel_chosen=(8,4)
def dE2lbd(dE):
    lbd0=dE*n
    return np.sqrt(np.exp(-lbd0)+np.exp(lbd0)-2)

def lbd2tf2ratio(dE):    
    lbd=dE2lbd(dE)
    
    tf2ratio={}
    for tf in tfs_conn:
        c3=key2tf2c3pt['gS+'][tf]
        c3=-(np.roll(c3,-n,axis=-1)+np.roll(c3,n,axis=-1)-2*c3) + lbd**2*c3
        c2=(lbd**2)*c2pt_disc[:,tf]
        tf2ratio[tf]=c3/c2[:,None]
    return tf2ratio
tfmins=[8,10,12,14,16]
tcmins=range(n+1,6+1)
fits_laplace=yu.doFits_3pt_lbd(lbd2tf2ratio,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'lbd_{n}'+extraLabel,verbose=1)
fits_laplace=[[(tfmin,tcmin),np.array([pars_jk[:,0],np.abs(pars_jk[:,1])]).T,chi2_jk,Ndof] for (tfmin,tcmin),pars_jk,chi2_jk,Ndof in fits_laplace]

fits_laplace_ds2=yu.doFits_3pt_lbd(lbd2tf2ratio,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'lbd_{n}_ds2'+extraLabel,verbose=1,downSampling=[1,2])
fits_laplace_ds2=[[(tfmin,tcmin),np.array([pars_jk[:,0],np.abs(pars_jk[:,1])]).T,chi2_jk,Ndof] for (tfmin,tcmin),pars_jk,chi2_jk,Ndof in fits_laplace_ds2]

fits_laplace_unc=yu.doFits_3pt_lbd(lbd2tf2ratio,tfmins[:1],tcmins[:1],symmetrizeQ=symmetrizeQ,label=f'lbd_{n}_unc'+extraLabel,corrQ=False,verbose=2)
fits_laplace_unc=[[(tfmin,tcmin),np.array([pars_jk[:,0],np.abs(pars_jk[:,1])]).T,chi2_jk,Ndof] for (tfmin,tcmin),pars_jk,chi2_jk,Ndof in fits_laplace_unc]

fit_MA_laplace=yu.doMA_3pt(fits_laplace,fitlabels=fitlabel_chosen)

print(yu.jackme_un2str(fit_MA_laplace[0][:,1]*yu.ens2aInv[ens]))

dE=np.mean(fit_MA_laplace[0][:,1])
tf2ratio_laplace=lbd2tf2ratio(dE)
tfmins=[8,10,12,14,16,18]
tcmins=range(n+1,6+1)
fits_const_2=yu.doFits_3pt('const',tf2ratio_laplace,tfmins,tcmins,symmetrizeQ=symmetrizeQ,label=f'const_2_laplace'+extraLabel)
fit_const_MA_2=yu.doMA_3pt(fits_const_2,fitlabels=fitlabel_chosen)

[verbose1] tfmin=8;
[verbose1] tfmin=10;
[verbose1] tfmin=12;
[verbose1] tfmin=14;
[verbose1] tfmin=16;
[verbose1] tfmin=8;
[verbose1] tfmin=10;
[verbose1] tfmin=12;
[verbose1] tfmin=14;
[verbose1] tfmin=16;
[verbose2] tfmin=8, tcmin=3;
347(24)


In [22]:
dic={
    'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio,None,None,None,None],
    'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
    'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
    'xyunit':[xunit,yunit],
    'mfc:[global]':['None'],
}
dic_lbd={
    'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio_laplace,None,fits_laplace,None,None],
    'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
    'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,n+1,None],
    'xyunit':[xunit,yunit],
    'mfc:[global]':['white'],
}
dic_lbd_2={
    'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,fits_const_2,None,None],
    'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
    'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,n+1,None],
    'xyunit':[xunit,yunit],
    'mfc:[global]':['white'],
}

label2dic={}
for label in label2fits.keys():
    label2dic[label]={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,None,label2fits[label]],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
        'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,5,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
        'xyunit':[xunit,yunit],
        'mfc:[global]':['None'],
    }

colHeaders=['ratio','fit_laplace','2st2step_SYM','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow'],colHeaders=colHeaders,fontsize_colHeaders=20)
fig,axs=yu.makePlot_3pt(dic_lbd,shows=['rainbow']+[None]*(colHeaders.index('fit_laplace')-1)+['fit_const'],figAxs=(fig,axs),colHeaders=None)
for i,label in enumerate(colHeaders):
    if label in label2dic.keys():
        fig,axs=yu.makePlot_3pt(label2dic[label],shows=['rainbow']+[None]*(i-1)+['fit_2st'],figAxs=(fig,axs),colHeaders=None)

fig,axs=yu.makePlot_3pt(label2dic['2st2step_SYMshare'],shows=['rainbow','fit_2st'],figAxs=(fig,axs),colHeaders=None)

axs[0,0].set_ylim([5,75])
yu.finalizePlot('lbdfit_vs_2st'+extraLabel)

In [23]:
dic_lbd_ds2={
    'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[tf2ratio_laplace,None,fits_laplace_ds2,None,None],
    'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
    'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,n+1,None],
    'xyunit':[xunit,yunit],
    'mfc:[global]':['white'],
}

label2dic_ds2={}
for label in label2fits_ds2.keys():
    label2dic_ds2[label]={
        'base:[tf2ratio,fits_band,fits_const,fits_sum,fits_2st]':[None,None,None,None,label2fits_ds2[label]],
        'WAMA:[fit_band_WA,fit_const_MA,fit_sum_MA,fit_2st_MA]':[None,None,None,None],
        'fit_2st:[tfmin_min,tfmin_max,tcmin_min,tcmin_max,dtf,dtc]':[None,None,None,5,None,None],
        'rainbow:[tfmin,tfmax,tcmin,dt]':[None,None,1,None],
        'xyunit':[xunit,yunit],
        'mfc:[global]':['None'],
    }

colHeaders=['ratio','fit_laplace','2st2step_SYM','2st2step_SYM_0ra11','2st2step_SYM_0rc1_0ra11','2st2step_SYM_share11']
fig,axs=yu.makePlot_3pt(dic,shows=['rainbow'],colHeaders=colHeaders,fontsize_colHeaders=20)
fig,axs=yu.makePlot_3pt(dic_lbd_ds2,shows=['rainbow']+[None]*(colHeaders.index('fit_laplace')-1)+['fit_const'],figAxs=(fig,axs),colHeaders=None)
for i,label in enumerate(colHeaders):
    if label in label2dic_ds2.keys():
        fig,axs=yu.makePlot_3pt(label2dic_ds2[label],shows=['rainbow']+[None]*(i-1)+['fit_2st'],figAxs=(fig,axs),colHeaders=None)

fig,axs=yu.makePlot_3pt(label2dic_ds2['2st2step_SYMshare'],shows=['rainbow','fit_2st'],figAxs=(fig,axs),colHeaders=None)

axs[0,0].set_ylim([5,75])
yu.finalizePlot('lbdfit_vs_2st_ds2'+extraLabel)